# Example: Bandit Challenger and Validation Report

In this example, we implement the **epsilon-greedy combinatorial bandit** for adaptive asset selection, backtest it against the Cobb-Douglas engine and buy-and-hold across all HMM-generated paths, and produce a formal validation report with pass/fail criteria for deployment readiness.

> **By the end of this example, you will be able to:**
> * Implement and run the bandit portfolio selector with the `world()` function
> * Compare three strategies head-to-head across 100 HMM regime-switching paths
> * Produce a validation report that determines which strategies are cleared for Session 4 production

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

In [ ]:
let
    # Load scenario and engine results from Example 1 -
    scenario_data = load(joinpath(_PATH_TO_DATA, "backtest-scenario.jld2"));
    global scenario = scenario_data["scenario"];

    results_data = load(joinpath(_PATH_TO_DATA, "backtest-results.jld2"));
    global engine_bt = results_data["engine"];
    global bh_bt = results_data["buyhold"];

    # Constants -
    global Δt = 1.0 / 252.0;
    global rf = 0.05;
    global B₀ = 10000.0;
    global N_short = 21;
    global N_long = 63;
    global offset = N_short + N_long;
    global n_mc_paths = scenario.n_paths;

    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    println("Loaded: $(n_mc_paths) paths, $(length(tickers)) tickers")
    println("Engine and buy-and-hold results loaded from Example 1")
end

___
## Task 1: Implement and Run the Bandit Portfolio Selector
We demonstrate the epsilon-greedy bandit on a single representative path. The bandit explores $2^K - 1 = 31$ possible asset subsets, using the Cobb-Douglas utility as its reward signal via the `bandit_world()` function. We visualize how the bandit converges to a preferred subset.

> **What should you see?** The reward history should show high variance initially (exploration) and stabilize as the bandit converges (exploitation). The exploration probability $\epsilon_t$ decays over time. The winning arm should be a sensible asset subset given the market conditions.

In [ ]:
let
    # Use the first path as a representative example -
    K = length(tickers);
    mkt = scenario.market_paths[1, :];

    # Compute lambda at offset -
    ema_s = compute_ema(mkt; window=N_short);
    ema_l = compute_ema(mkt; window=N_long);
    λ = compute_lambda(ema_s, ema_l; G=10.0);

    # Compute market growth -
    gm_raw = compute_market_growth(mkt; Δt=Δt);
    gm_e = compute_ema(gm_raw; window=10);

    prices_at_offset = [scenario.price_paths[1, offset, k] for k in 1:K];

    # Build bandit context -
    bandit_ctx = build(MyBanditContext, (
        tickers = tickers, sim_parameters = sim_params,
        prices = prices_at_offset, B = B₀,
        gm_t = gm_e[min(offset, length(gm_e))],
        lambda = λ[offset], epsilon = 0.1
    ));

    # Run bandit -
    bandit_model = build(MyEpsilonGreedyBanditModel, (
        K = K, n_iterations = 500, alpha = 0.1
    ));

    global bandit_result = solve_bandit(bandit_model, bandit_ctx);

    # Report -
    println("Bandit converged after 500 iterations:")
    println("  Best action: $(bandit_result.best_action)")
    selected_tickers = tickers[bandit_result.best_action .== 1];
    excluded_tickers = tickers[bandit_result.best_action .== 0];
    println("  Selected assets: $(selected_tickers)")
    println("  Excluded assets: $(excluded_tickers)")
    println("  Best utility: $(round(bandit_result.best_utility, digits=4))")
end

**Visualize:** Bandit convergence — reward history and exploration decay.

In [ ]:
let
    T_bandit = length(bandit_result.reward_history);
    iters = 1:T_bandit;

    # Top: reward history with running average -
    window = 20;
    running_avg = [mean(bandit_result.reward_history[max(1, t-window+1):t]) for t in 1:T_bandit];

    p1 = scatter(iters, bandit_result.reward_history, label="Reward", alpha=0.2,
        markersize=2, color=:steelblue)
    plot!(p1, iters, running_avg, label="Running avg ($(window))", linewidth=2, color=:coral)
    ylabel!(p1, "Cobb-Douglas Utility")
    title!(p1, "Bandit Reward History")

    # Bottom: exploration probability -
    p2 = plot(iters, bandit_result.exploration_history, label="εₜ", linewidth=2, color=:purple)
    xlabel!(p2, "Iteration")
    ylabel!(p2, "Exploration Prob.")
    title!(p2, "Exploration Decay")

    plot(p1, p2, layout=(2, 1), size=(800, 500), legend=:topright)
end

**Visualize:** Cumulative regret — how much reward was lost to exploration.

In [ ]:
let
    regret = compute_regret(bandit_result.reward_history);
    plot(1:length(regret), regret, label="Cumulative Regret", linewidth=2, color=:coral)
    xlabel!("Iteration")
    ylabel!("Cumulative Regret")
    title!("Bandit Cumulative Regret")
    plot!(size=(800, 350))
end

___
## Task 2: Backtest Bandit-Selected Portfolios Across All Paths
We run the bandit selector across all HMM-generated paths and compare three strategies head-to-head: the Cobb-Douglas engine (all assets, utility allocation), the bandit selector (learned asset subset + Cobb-Douglas allocation), and buy-and-hold.

> **What should you see?** A comparison table with 6 metrics across 3 strategies. The bandit may outperform if it successfully excludes assets that hurt during regime shifts. It may underperform if its initial learning phase costs too much. The key question: does adaptive asset _selection_ add value over adaptive asset _weighting_?

In [ ]:
let
    # Backtest the bandit across all paths -
    println("Backtesting Bandit Selector across $(n_mc_paths) paths...")
    global bandit_bt = backtest_bandit(scenario, tickers, sim_params;
        B₀=B₀, offset=offset, n_bandit_iters=200);

    println("Bandit backtest complete.")
    println("Median final wealth: \$$(round(median(bandit_bt.final_wealth), digits=0))")
end

**Head-to-head comparison table.**

In [ ]:
let
    # Compute comparison metrics -
    function strategy_metrics(bt::MyBacktestResult)
        return (
            med_wealth = round(median(bt.final_wealth), digits=0),
            p10_wealth = round(quantile(bt.final_wealth, 0.10), digits=0),
            p90_wealth = round(quantile(bt.final_wealth, 0.90), digits=0),
            med_dd = round(median(bt.max_drawdowns) * 100, digits=1),
            med_sharpe = round(median(bt.sharpe_ratios), digits=3),
            fail_rate = round(sum(bt.final_wealth .< B₀) / length(bt.final_wealth) * 100, digits=1)
        )
    end

    eng = strategy_metrics(engine_bt);
    ban = strategy_metrics(bandit_bt);
    bh = strategy_metrics(bh_bt);

    comparison = DataFrame(
        "Metric" => ["Median Final Wealth", "10th Percentile Wealth",
            "90th Percentile Wealth", "Median Max Drawdown (%)",
            "Median Sharpe Ratio", "Failure Rate (%)"],
        "Cobb-Douglas Engine" => [eng.med_wealth, eng.p10_wealth, eng.p90_wealth,
            eng.med_dd, eng.med_sharpe, eng.fail_rate],
        "Bandit Selector" => [ban.med_wealth, ban.p10_wealth, ban.p90_wealth,
            ban.med_dd, ban.med_sharpe, ban.fail_rate],
        "Buy-and-Hold" => [bh.med_wealth, bh.p10_wealth, bh.p90_wealth,
            bh.med_dd, bh.med_sharpe, bh.fail_rate]
    );

    println("═"^70)
    println("  HEAD-TO-HEAD: 3 Strategies × $(n_mc_paths) HMM Paths")
    println("═"^70)
    pretty_table(comparison, tf=tf_markdown)
end

**Visualize:** Box plots — final wealth and max drawdown distributions.

In [ ]:
let
    p1 = boxplot(["CD Engine" "Bandit" "Buy-Hold"],
        [engine_bt.final_wealth bandit_bt.final_wealth bh_bt.final_wealth],
        legend=false, ylabel="Final Wealth (\$)", title="Final Wealth Distribution",
        color=[:steelblue :green :coral], alpha=0.7)
    hline!(p1, [B₀], linestyle=:dash, color=:black, alpha=0.5, label="")

    p2 = boxplot(["CD Engine" "Bandit" "Buy-Hold"],
        [engine_bt.max_drawdowns.*100 bandit_bt.max_drawdowns.*100 bh_bt.max_drawdowns.*100],
        legend=false, ylabel="Max Drawdown (%)", title="Max Drawdown Distribution",
        color=[:steelblue :green :coral], alpha=0.7)

    plot(p1, p2, layout=(1, 2), size=(900, 400))
end

___
## Task 3: Formal Validation Report
We evaluate each strategy against explicit pass/fail criteria for production deployment:

| Criterion | Threshold |
|-----------|----------|
| Median Sharpe | $\geq$ 0.3 |
| Median Max Drawdown | $\leq$ 25% |
| Failure Rate | $\leq$ 10% |
| Beats Buy-and-Hold | Median wealth $>$ buy-and-hold median |

Strategies that pass all four criteria are cleared for Session 4 production deployment.

> **What should you see?** A pass/fail table for each strategy. The Cobb-Douglas engine should pass most criteria (it has built-in risk controls). The bandit's performance depends on whether adaptive selection adds value in this scenario. Buy-and-hold will likely fail on drawdown or failure rate.

In [ ]:
let
    bh_median_wealth = median(bh_bt.final_wealth);

    # Define criteria -
    criteria = Dict(
        "min_sharpe" => 0.3,
        "max_drawdown" => 0.25,
        "max_failure_rate" => 0.10,
        "min_wealth_vs_bh" => bh_median_wealth
    );

    # Build reports for each strategy -
    strategies = [
        ("Cobb-Douglas Engine", engine_bt),
        ("Bandit Selector", bandit_bt),
        ("Buy-and-Hold", bh_bt)
    ];

    global reports = [];
    for (label, bt) in strategies
        actuals = Dict(
            "min_sharpe" => median(bt.sharpe_ratios),
            "max_drawdown" => median(bt.max_drawdowns),
            "max_failure_rate" => sum(bt.final_wealth .< B₀) / length(bt.final_wealth),
            "min_wealth_vs_bh" => median(bt.final_wealth)
        );

        report = build(MyValidationReport;
            strategy_label=label, criteria=criteria, actuals=actuals);
        push!(reports, report);
    end

    # Display validation results -
    println("═"^70)
    println("  VALIDATION REPORT: Production Readiness")
    println("═"^70)

    for report in reports
        println("\n  Strategy: $(report.strategy_label)")
        println("  " * "-"^40)

        criterion_names = Dict(
            "min_sharpe" => "Sharpe ≥ 0.3",
            "max_drawdown" => "Max DD ≤ 25%",
            "max_failure_rate" => "Failure ≤ 10%",
            "min_wealth_vs_bh" => "Beats Buy-Hold"
        );

        all_passed = true;
        for (key, passed) in report.passed
            status = passed ? "PASS" : "FAIL";
            actual = round(report.actuals[key], digits=3);
            threshold = round(report.criteria[key], digits=3);
            name = get(criterion_names, key, key);
            println("    $(name): $(status) (actual=$(actual), threshold=$(threshold))")
            if !passed
                all_passed = false;
            end
        end
        verdict = all_passed ? "CLEARED for Session 4" : "NOT CLEARED";
        println("  Verdict: $(verdict)")
    end
end

**Summary table:** Pass/fail at a glance.

In [ ]:
let
    # Build summary DataFrame -
    criterion_keys = ["min_sharpe", "max_drawdown", "max_failure_rate", "min_wealth_vs_bh"];
    criterion_labels = ["Sharpe ≥ 0.3", "Max DD ≤ 25%", "Failure ≤ 10%", "Beats Buy-Hold"];

    rows = [];
    for (i, label) in enumerate(criterion_labels)
        row = [label];
        for report in reports
            key = criterion_keys[i];
            passed = get(report.passed, key, false);
            push!(row, passed ? "PASS" : "FAIL");
        end
        push!(rows, row);
    end

    # Add overall verdict -
    verdict_row = ["OVERALL"];
    for report in reports
        all_pass = all(values(report.passed));
        push!(verdict_row, all_pass ? "CLEARED" : "NOT CLEARED");
    end
    push!(rows, verdict_row);

    data = reduce(hcat, rows)';
    pretty_table(data,
        header=["Criterion", [r.strategy_label for r in reports]...],
        alignment=[:l, :c, :c, :c])
end

___
## Summary
In this example, we introduced the **epsilon-greedy bandit** as an adaptive challenger to the Cobb-Douglas engine and produced a formal validation report:

- The **bandit** learns which asset subsets produce the highest Cobb-Douglas utility through trial-and-error exploration
- **Head-to-head backtesting** across 100 HMM paths reveals whether adaptive _selection_ (bandit) adds value over adaptive _weighting_ (Cobb-Douglas engine)
- The **validation report** applies explicit pass/fail criteria — strategies that pass are cleared for production deployment in Session 4

In Session 4, the bandit and Cobb-Douglas allocator will work **together**: the bandit selects assets continuously, and the allocator decides how much of each. We'll add sentiment monitoring, alert systems, and human override protocols.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.